# 2. Anthropic Claude generation

**Learning objective:** understand how a validated `LearningPathway` is produced
from a family intake plus supplied context, how the provider boundary is kept
small, and why generation was proved before retrieval existed.

**Where this fits:** this is the right-hand end of the pipeline.

```
intake -> [retrieval] -> context -> prompt -> Claude -> LearningPathway
```

The bracketed step does not exist yet in this notebook. Context is assembled by
hand so that generation can be judged on its own.

## Why generation was proved first

A RAG system has two independent risks: retrieving the wrong evidence, and
writing badly from good evidence. If both are built at once, every bad result is
ambiguous.

So the first working slice of this project used a small set of hand-picked
passages as context. That answered one question in isolation: *given good
evidence, can the model produce something a family would actually want to read,
in a schema we can validate?* Only after that was answered did retrieval get
built.

In [ ]:
import inspect

from pydantic import ValidationError

from mosaic_pathway.generation import ClaudePathwayGenerator
from mosaic_pathway.models import (
    ChildProfile,
    FamilyIntake,
    LearningPathway,
)
from mosaic_pathway.prompts import SYSTEM_PROMPT, build_generation_prompt
from mosaic_pathway.settings import Settings, load_settings

print("generation modules imported")

In [ ]:
intake = FamilyIntake(
    children=[
        ChildProfile(
            label="older child",
            age=12,
            interests=["animals", "drawing"],
            learning_needs=["movement breaks"],
        )
    ],
    leaving_behind=["rigid daily schedules"],
    wants_to_preserve=["reading together after dinner"],
    wants_to_add=["more time outdoors"],
    family_values=["curiosity", "gentleness"],
    practical_constraints=["one working parent at home"],
    additional_context="We are in our first year of self-directed learning.",
)

manual_context = [
    {
        "source_id": "synthetic-guide-0007",
        "title": "Synthetic guide, chunk 7",
        "text": (
            "Families often start with one small repeated practice rather than a "
            "full timetable, and let the rhythm grow from what already works."
        ),
    },
    {
        "source_id": "synthetic-guide-0011",
        "title": "Synthetic guide, chunk 11",
        "text": (
            "An interest catalog is a simple shared list of what a child keeps "
            "returning to, used to choose the next resource together."
        ),
    },
]

print("context passages:", len(manual_context))
print("context keys:", sorted(manual_context[0]))

## System prompt versus user prompt

The split is deliberate and stable:

* the **system prompt** carries the role, the tone, and the rules that never
  change between families, including the rule that every recommendation must
  come from the supplied context
* the **user prompt** carries only the variable payload: this family's intake and
  this family's context

Keeping the rules out of the per-request payload means they cannot drift, and it
makes the request easier to reason about when a result looks wrong.

Claude enforces that split at the API level. The Messages API has no `"system"`
role inside the message list. The rules travel in a separate top-level `system`
parameter, and `messages` holds only the user turn. That is a real difference
from the OpenAI-shaped chat APIs, where the system prompt is the first message.

In [ ]:
system_lines = SYSTEM_PROMPT.splitlines()

print("system prompt lines:", len(system_lines))
print("system prompt characters:", len(SYSTEM_PROMPT))
print()
print("\n".join(system_lines[:6]))

In [ ]:
user_prompt = build_generation_prompt(intake, manual_context)

print("user prompt characters:", len(user_prompt))
print()
print(user_prompt[:300])
print("...")
print()
print("sections:", [line for line in user_prompt.splitlines() if line.isupper()])

## Structured output instead of free-form JSON

There are three common ways to get structured data out of a language model:

| Approach | What can go wrong |
| --- | --- |
| Ask for JSON in the prose | Fences, commentary, trailing text, invented fields |
| JSON mode | Valid JSON, but not necessarily the right shape |
| Structured output from a schema | The provider is constrained by the schema itself |

This project uses the third option. `client.messages.parse` is given
`output_format=LearningPathway`, so the same Pydantic model from notebook 1 is
both the schema sent to Claude and the type returned on `response.parsed_output`.

There is no `json.loads`, no regular expression, no fence stripping, and no
tool-use workaround anywhere in the generation path.

The contract still gets validated on the way back, because a provider can decline
to produce a parsed result.

In [ ]:
incomplete_response = {
    "family_reflection": "A short reflection.",
    "starting_rhythm": [
        {
            "timing": "Most mornings",
            "practice": "Take a short walk.",
            "why_it_fits": "It adds outdoor time.",
        }
    ],
    "resources": [],
    "closing_note": "Go gently.",
}

try:
    LearningPathway.model_validate(incomplete_response)
except ValidationError as error:
    for item in error.errors():
        print(list(item["loc"]), "->", item["msg"])

Three separate failures are reported at once: too few rhythm practices, too few
resources, and a missing community suggestion. None of those would have been
caught by a `json.loads` call. This is why the schema is the contract rather than
a suggestion in the prompt.

## Authentication and configuration

Claude is reached with an Anthropic API key. The key is created in the Anthropic
Console and read from a local `.env` file, never from a literal in the source.

Three values are configured, and all of them are read through a settings model
rather than scattered `os.environ` lookups. The key is held as a Pydantic
`SecretStr`, so printing the settings object cannot leak it.

In [ ]:
for name, field in Settings.model_fields.items():
    requirement = "required" if field.is_required() else "optional"
    print(f"{name:<22} <- environment alias {field.alias:<22} ({requirement})")

print()
print("settings loader:", load_settings.__name__)

The values themselves are not printed anywhere in this notebook.

Note that `ANTHROPIC_MODEL` is required and has no default. No model identifier
is hard-coded in the source, the tests, or the documentation, so adopting a newer
Claude model is a one-line edit in `.env` rather than a code change.

## The generator boundary

Everything provider-specific lives behind one class with one method. That method
signature is the only thing the rest of the system knows about Claude.

The optional `client` parameter exists so tests and notebooks can pass a fake
client and exercise the class without a network call. It is not a provider
abstraction: Claude is the single provider on this branch, and there is no base
class, registry, or factory.

In [ ]:
print("class :", ClaudePathwayGenerator.__name__)
print("init  :", inspect.signature(ClaudePathwayGenerator.__init__))
print("call  :", inspect.signature(ClaudePathwayGenerator.generate))

## Watching the request shape without calling the API

A fake client records exactly what the generator would send. This is how the test
suite proves that the rules go in the top-level `system` parameter, that
`messages` holds only the user turn, and that `LearningPathway` is passed as the
output format.

In [ ]:
class RecordingMessages:
    def __init__(self) -> None:
        self.calls: list[dict] = []

    def parse(self, **kwargs):
        self.calls.append(kwargs)
        raise RuntimeError("no network call is made in this notebook")


class RecordingClient:
    def __init__(self) -> None:
        self.messages = RecordingMessages()


fake_client = RecordingClient()
settings = Settings(
    ANTHROPIC_API_KEY="notebook-placeholder-key",
    ANTHROPIC_MODEL="notebook-placeholder-model",
    _env_file=None,
)
generator = ClaudePathwayGenerator(settings, client=fake_client)

try:
    generator.generate(intake, manual_context)
except RuntimeError as error:
    print("stopped before the network:", error)

request = fake_client.messages.calls[0]

print()
print("request keys      :", sorted(request))
print("message roles     :", [message["role"] for message in request["messages"]])
print("system is separate:", request["system"] is SYSTEM_PROMPT)
print("output format     :", request["output_format"].__name__)
print("max tokens        :", request["max_tokens"])

That narrow boundary buys three things:

* the RAG service in notebook 5 depends on a callable shape, not on a vendor SDK,
  so it can be exercised offline with a small fake
* swapping providers means writing one new class, not editing the pipeline
* failures are localized: a missing parsed output is raised as a `RuntimeError`
  at the boundary rather than leaking a provider object upward

## Provider-specific failures

The Anthropic SDK raises typed exceptions, so failures are classified by type
rather than by matching words in an error string. The API layer and the Streamlit
layer each translate those types into their own vocabulary.

| Exception | Meaning | API response |
| --- | --- | --- |
| `AuthenticationError`, `PermissionDeniedError` | The key is missing, revoked, or lacks model access | 503 `anthropic_authentication_failed` |
| `RateLimitError` | The account hit an Anthropic rate limit | 503 `anthropic_rate_limited` |
| `APIConnectionError`, `APITimeoutError` | The API could not be reached in time | 503 `anthropic_unreachable` |
| any other `AnthropicError` | Something else went wrong provider-side | 502 `pathway_generation_failed` |

The application does not retry. The SDK's own transport-level retry is the only
retry in the system, and provider-side error text is never shown to a family.

In [ ]:
from anthropic import (
    AnthropicError,
    APIConnectionError,
    APITimeoutError,
    AuthenticationError,
    PermissionDeniedError,
    RateLimitError,
)

for exception in (
    AuthenticationError,
    PermissionDeniedError,
    RateLimitError,
    APIConnectionError,
    APITimeoutError,
):
    print(
        f"{exception.__name__:<22} is an AnthropicError:",
        issubclass(exception, AnthropicError),
    )

print()
print(
    "APITimeoutError is a connection error:",
    issubclass(APITimeoutError, APIConnectionError),
)

## Optional live Claude section

Everything above runs offline. The cells below are the only ones in this notebook
that contact the Anthropic API, and they are disabled by default. Running them
spends money.

To run them you need:

1. an Anthropic API key with access to a model that supports structured outputs
2. `ANTHROPIC_API_KEY` and `ANTHROPIC_MODEL` set in a local `.env` file
3. network access from the shell that started Jupyter

Set the flag below to `True` only when all three are true. The output prints the
family-facing pathway only.

In [ ]:
RUN_LIVE_CLAUDE = False

print("live Claude section enabled:", RUN_LIVE_CLAUDE)

In [ ]:
if RUN_LIVE_CLAUDE:
    live_generator = ClaudePathwayGenerator(load_settings())
    live_pathway = live_generator.generate(intake, manual_context)

    print(live_pathway.family_reflection)
    print()

    for practice in live_pathway.starting_rhythm:
        print(f"{practice.timing}: {practice.practice}")

    print()

    for resource in live_pathway.resources:
        print(f"{resource.title} [{resource.source_id}]")

    print()
    print(live_pathway.closing_note)
else:
    print("Skipped: set RUN_LIVE_CLAUDE to True to call the Anthropic API.")

## The limitation this slice does not solve

A returned pathway carries a `source_id` on every resource. It is tempting to
read that as proof the recommendation is supported by that passage.

It is not. At this stage the id is only a string the model copied from the
context it was given. Nothing yet checks that the id was ever retrieved, and
nothing at all checks that the passage semantically supports the claim.

Notebook 5 adds the first of those checks. The second one is still, honestly, a
human job, which is why notebook 6 keeps a review rubric alongside the automated
suite.

## Key takeaways

* Generation was proved with hand-assembled context so that retrieval quality and
  writing quality could be judged separately.
* The system prompt holds invariant rules and travels in Claude's top-level
  `system` parameter; the user prompt holds only this family's payload.
* Structured output makes the Pydantic schema the contract, and validation still
  runs on the way back through `response.parsed_output`.
* Authentication is an Anthropic API key, and the key, the model, and the token
  ceiling are all read through a settings model.
* All provider-specific code sits behind one class with one method.

## Next

Notebook 3 replaces the hand-assembled context with a real knowledge base built
from documents, which is where retrieval quality is actually decided.